<a href="https://colab.research.google.com/github/matteogolfarelli/DataCentricAI/blob/main/Data%20Preparation/Data_Preparation_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Centric AI — Lab: Data Preparation in Practice

**Course:** Data Centric AI — Module I (Prof. M. Golfarelli)
**Topic:** Data Preparation (CRISP-DM, step 3)

This lab puts into practice the techniques of the *Data Preparation* slide deck. We keep working on the **Titanic** dataset already explored in the Data Understanding lab, and we use the **Car Evaluation** dataset of the slides for the encoding part.

| Part | Content | Data | Slides |
|---|---|---|---|
| 1 | Encoding categorical data | Car Evaluation, Titanic | 3–16 |
| 2 | Data cleaning and missing values | Titanic | 17–26 |
| 3 | Scaling | Titanic | 36–47 |
| 4 | Sampling and coverage bias | Titanic | 48–62 |
| 5 | Mitigating a coverage bias: collecting data vs oversampling | Titanic | 59, 65 |
| 6 | Wrap-up: an end-to-end preparation pipeline | Titanic | — |


The class attribute of Titanic is always drawn with the same colors: <span style="color:#D55E00">■ **0 = Died**</span> &nbsp; <span style="color:#0072B2">■ **1 = Survived**</span>

## Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, RepeatedStratifiedKFold, cross_val_score, cross_validate, cross_val_predict
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, MinMaxScaler, RobustScaler, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, recall_score, classification_report

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid", context="notebook")
CLASS = "Survived"
PALETTE = {0: "#D55E00", 1: "#0072B2"}
ENC_COLORS = {"Label (alphabetical)": "#7F7F7F", "Ordinal (natural order)": "#2E9E8F", "One-hot": "#ED7D31"}
RANDOM_STATE = 42

As in every ML project, we first put aside a **test set** (20% of the passengers, with the same share of survivors as the whole dataset). All preparation steps (imputers, scalers, encoders) are **learnt on the training set only** and then applied to new data, exactly as they will be applied in production (slide 26).

In [ ]:
URL = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(URL)

# 20% of the passengers are put aside as a test set; every preparation step is learnt on the training set only
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df[CLASS], random_state=RANDOM_STATE)
train_df, test_df = train_df.reset_index(drop=True), test_df.reset_index(drop=True)
print(f"training set: {len(train_df)} passengers   test set: {len(test_df)} passengers")

---
# Part 1 — Encoding categorical data

## 1.1 Label vs one-hot vs ordinal encoding on Car Evaluation (slides 4–6, 14–15)

The **Car Evaluation** dataset (slide 14) describes 1,728 cars with six **ordinal** attributes (e.g. `buying` = low, med, high, vhigh); the class (`unacc`, `acc`, `good`, `vgood`) is ordinal too. We compare three ways of turning the categories into numbers:
* **Label encoding**: each category becomes an integer in *alphabetical* order (`high` = 0, `low` = 1, `med` = 2, `vhigh` = 3);
* **Ordinal encoding**: each category becomes an integer that respects the *natural order* (`low` = 0, `med` = 1, `high` = 2, `vhigh` = 3) (slide 5);
* **One-hot encoding**: each category becomes a binary column (slide 4).

and two classifiers: a **Decision Tree** and a **KNN** (k = 5 neighbours).

**How do we measure the results?** We use 5-fold cross-validation and the three indicators of slide 15:
* **Accuracy**: share of cars classified correctly.
* **Recall of a class**: share of the cars *of that class* that the model recognises. E.g., if 45 of the 65 `vgood` cars are predicted `vgood`, the recall of `vgood` is 45/65 = 0.69.
* **Precision of a class**: share of the cars *predicted* in that class that really belong to it.

With 4 classes we get 4 recalls (and 4 precisions). **Macro-averaged recall** is simply their *average*: each class counts 1/4, whatever its size. This matters here because 70% of the cars are `unacc`: accuracy is dominated by this class, while macro recall gives the small classes `good` (69 cars) and `vgood` (65 cars) the same weight as `unacc`. A model that ignores the small classes can have a high accuracy but a low macro recall.

In [ ]:
CAR_COLS = ["buying", "maint", "doors", "persons", "lug_boot", "safety", "class"]
ORDER = {"buying": ["low", "med", "high", "vhigh"], "maint": ["low", "med", "high", "vhigh"],
         "doors": ["2", "3", "4", "5more"], "persons": ["2", "4", "more"],
         "lug_boot": ["small", "med", "big"], "safety": ["low", "med", "high"]}
CLASS_ORDER = ["unacc", "acc", "good", "vgood"]
try:   # the file of slide 14
    car = pd.read_csv("https://raw.githubusercontent.com/matteogolfarelli/DataCentricAI/main/Data%20Understanding/CarEvaluation.csv", dtype=str)
except Exception:
    car = pd.read_csv("https://archive.ics.uci.edu/ml/machine-learning-databases/car/car.data", names=CAR_COLS, dtype=str)
print(car.shape)
display(car.head())
car["class"].value_counts().reindex(CLASS_ORDER).to_frame("number of cars").T

In the table below each row is an encoding; for each model you find the three indicators. Darker green = better within the column, and the best value of each column is in bold with a frame. The chart shows the same numbers.

In [ ]:
features = CAR_COLS[:-1]
Xc, yc = car[features], car["class"]

encoders = {
    "Label (alphabetical)": OrdinalEncoder(),                                        # categories numbered in alphabetical order
    "Ordinal (natural order)": OrdinalEncoder(categories=[ORDER[c] for c in features]),
    "One-hot": OneHotEncoder(),
}
print("Label encoding of 'buying':  ", dict(zip(sorted(ORDER["buying"]), range(4))))
print("Ordinal encoding of 'buying':", dict(zip(ORDER["buying"], range(4))))

models = {"Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE), "KNN (k=5)": KNeighborsClassifier(n_neighbors=5)}
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
METRICS = {"Accuracy": "accuracy", "Precision (macro)": "precision_macro", "Recall (macro)": "recall_macro"}
rows = []
for enc_name, enc in encoders.items():
    for m_name, m in models.items():
        res = cross_validate(make_pipeline(enc, m), Xc, yc, cv=cv5, scoring=METRICS)
        rows.append({"Encoding": enc_name, "Model": m_name, **{k: res[f"test_{k}"].mean() for k in METRICS}})
long = pd.DataFrame(rows)
enc_results = long.pivot(index="Encoding", columns="Model", values=list(METRICS)).swaplevel(axis=1).sort_index(axis=1, level=0)
enc_results = enc_results[[(m, k) for m in models for k in METRICS]].reindex(list(encoders))

# table: one row per encoding, darker green = better, best value of each column in bold
display(enc_results.style.format("{:.3f}").background_gradient(cmap="Greens", axis=0)
        .highlight_max(axis=0, props="font-weight: bold; border: 2px solid black")
        .set_caption("5-fold cross-validation on Car Evaluation (mean of the 5 folds)"))

# the same numbers as a chart
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, m_name in zip(axes, models):
    d = long[long["Model"] == m_name].set_index("Encoding")[list(METRICS)].reindex(list(encoders))
    d.T.plot(kind="bar", ax=ax, color=[ENC_COLORS[e] for e in d.index], rot=0, width=.8, legend=(ax is axes[0]))
    for cont in ax.containers:
        ax.bar_label(cont, fmt="%.2f", fontsize=8)
    ax.set_title(m_name); ax.set_ylim(0.5, 1.05)
axes[0].legend(title="Encoding", loc="lower left", fontsize=9)
plt.tight_layout(); plt.show()

To see what macro recall hides, the next table shows the recall of **each class** for KNN: the macro recall in the last-but-one column is just the average of the four numbers on the left.

In [ ]:
# What does "macro-averaged recall" mean? Recall of EACH class for KNN, then their simple average
K_DEMO = 5          #  number of neighbours used in this cell
per_class = {}
for enc_name, enc in encoders.items():
    pred = cross_val_predict(make_pipeline(enc, KNeighborsClassifier(n_neighbors=K_DEMO)), Xc, yc, cv=cv5)
    rec = recall_score(yc, pred, labels=CLASS_ORDER, average=None)
    per_class[enc_name] = list(rec) + [rec.mean(), accuracy_score(yc, pred)]
cols = [f"{c} (n={n})" for c, n in yc.value_counts().reindex(CLASS_ORDER).items()] + ["macro recall = average", "accuracy"]
per_class = pd.DataFrame(per_class, index=cols).T
display(per_class.style.format("{:.2f}").background_gradient(cmap="RdYlGn", vmin=0.3, vmax=1.0)
        .set_caption(f"KNN (k={K_DEMO}): recall of each class = share of the cars of that class predicted correctly"))

In [ ]:
# number of input columns produced by each encoding
for enc_name, enc in encoders.items():
    print(f"{enc_name:25s} → {enc.fit_transform(Xc).shape[1]} columns")

The number of neighbours `k` is a parameter of KNN. The next cell repeats the evaluation of KNN for k = 1 … 30 and draws accuracy and macro recall for the three encodings (it takes about 30 seconds).

In [ ]:
# Effect of the number of neighbours k on KNN, for the three encodings
K_VALUES = list(range(1, 31))
sweep = []
for enc_name, enc in encoders.items():
    for k in K_VALUES:
        res = cross_validate(make_pipeline(enc, KNeighborsClassifier(n_neighbors=k)), Xc, yc, cv=cv5,
                             scoring={"Accuracy": "accuracy", "Recall (macro)": "recall_macro"})
        sweep.append({"Encoding": enc_name, "k": k, "Accuracy": res["test_Accuracy"].mean(), "Recall (macro)": res["test_Recall (macro)"].mean()})
sweep = pd.DataFrame(sweep)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, metric in zip(axes, ["Accuracy", "Recall (macro)"]):
    for enc_name, g in sweep.groupby("Encoding"):
        ax.plot(g["k"], g[metric], "o-", ms=3, color=ENC_COLORS[enc_name], label=enc_name)
    ax.axvline(5, ls=":", c="k"); ax.text(5.3, ax.get_ylim()[0] + 0.01, "k = 5 (table above)", fontsize=8)
    ax.set_xlabel("k (number of neighbours)"); ax.set_title(f"KNN — {metric}")
axes[0].legend()
plt.tight_layout(); plt.show()
sweep.pivot(index="k", columns="Encoding", values="Recall (macro)").loc[[1, 3, 5, 9, 15, 25]].round(3)

### Questions — encoding on Car Evaluation
1. Which encoding gives the best results for each model? Is the ranking the same as in slide 15?
2. Look at how label encoding maps `buying`. Which wrong "facts" about the price does KNN see when it computes distances? Why is the Decision Tree much less affected?
3. With one-hot encoding the distance between *any* two different categories of the same attribute is the same (e.g. `low`–`med` and `low`–`vhigh`). Why does this hurt KNN on this dataset in particular?
4. Look at the per-class recall table. Which classes make the macro recall of KNN with one-hot encoding so low? Why does accuracy not reveal the problem?
5.  Look at the chart of accuracy and macro recall as a function of `k`.
   * Is ordinal encoding the best for every value of `k`?
   * What happens to the macro recall when `k` is very small (1–3) and when it is large (≥ 20)?
   * Now set `K_DEMO = 1` and then `K_DEMO = 25` in the per-class recall cell and re-run it: which classes lose recall when `k` is large? Explain why, thinking of how many cars of each class there are.

## 1.2 Encoding the Titanic attributes

The Titanic attributes require different encodings depending on their type (Data Understanding lab, Part 1):

In [ ]:
tr = train_df.copy()
enc_demo = pd.DataFrame({
    "Sex → binary": (tr["Sex"] == "female").astype(int),
    "Pclass → ordinal (kept as 1,2,3)": tr["Pclass"],
})
emb = pd.get_dummies(tr["Embarked"], prefix="Emb").astype(int)       # nominal → one-hot
display(pd.concat([tr[["Sex", "Pclass", "Embarked"]], enc_demo, emb], axis=1).head(6))

# High-cardinality attributes
for c in ["Name", "Ticket", "Cabin"]:
    print(f"{c:7s}: {tr[c].nunique():4d} distinct values on {len(tr)} rows → one-hot would add {tr[c].nunique()} columns")

High-cardinality attributes are not one-hot encoded "as is". A typical alternative is to **extract a meaningful part** of the value or to **combine attributes** into a new one:
* `Title` from `Name` (Mr, Mrs, Miss, Master; the infrequent ones are grouped in `Rare`);
* `Deck` from the first letter of `Cabin`;
* `FamilySize` = `SibSp` + `Parch` + 1 (the passenger plus the relatives on board).

 A derived attribute must be computable **for one new passenger at a time**, using only the values of that passenger's record: in production the model will classify passengers one by one.

In [ ]:
def add_title(d):
    t = d["Name"].str.extract(r",\s*([^\.]+)\.")[0].replace({"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
    return t.where(t.isin(["Mr", "Mrs", "Miss", "Master"]), "Rare")             # group the rare titles

title = add_title(tr)                                                              # from Name
deck = tr["Cabin"].str[0].fillna("Unknown")                                        # first letter of Cabin
family_size = tr["SibSp"] + tr["Parch"] + 1                                        # from SibSp and Parch (+1 = the passenger)

derived = pd.DataFrame({"Title": title, "Deck": deck, "FamilySize": family_size.clip(upper=7), CLASS: tr[CLASS]})
for c in ["Title", "Deck", "FamilySize"]:
    display(derived.groupby(c)[CLASS].agg(survival_rate="mean", n="size").round(2).T.style.set_caption(c + ("  (7 = 7 or more)" if c == "FamilySize" else "")))

###  Questions — encoding Titanic
1. Why is `Embarked` one-hot encoded while `Pclass` can be kept as 1, 2, 3? Would one-hot encoding of `Pclass` be wrong?
2. `Title` has 17 distinct values in the full dataset; above, the infrequent ones are grouped in `Rare`. Why is it useful to group rare categories before one-hot encoding? What happens at prediction time if a new passenger has a title never seen in training (look at the parameter `handle_unknown` of `OneHotEncoder`)?
3. `FamilySize` and "number of passengers travelling on the same ticket" both describe the group a passenger travels with. `FamilySize` can be computed from the record of a single passenger, the second one cannot: why? Which survival pattern does `FamilySize` show?
4. Is `Deck = Unknown` a value or a missing value? Which choice would you make, given what we know about how cabins were recorded?

---
# Part 2 — Data cleaning and missing values

## 2.1 Quality checks (slides 17–18)

We run some simple checks on the whole dataset; the second cell shows the rows behind two of them.

In [ ]:
checks = {
    "duplicated rows": int(df.duplicated().sum()),
    "duplicated rows ignoring PassengerId": int(df.drop(columns="PassengerId").duplicated().sum()),
    "duplicated names": int(df["Name"].duplicated().sum()),
    "Fare = 0": int((df["Fare"] == 0).sum()),
    "Age < 1 (fractional ages)": int((df["Age"] < 1).sum()),
    "Age of the form xx.5 (estimated)": int(((df["Age"] % 1) == 0.5).sum()),
    "minors (Age < 14) travelling with SibSp + Parch = 0": int(((df["Age"] < 14) & (df["SibSp"] + df["Parch"] == 0)).sum()),
    "passengers sharing a ticket with a different Fare": int((df.groupby("Ticket")["Fare"].transform("nunique") > 1).sum()),
}
display(pd.Series(checks, name="count").to_frame())
# completeness
pd.DataFrame({"missing": df.isna().sum(), "missing %": (100 * df.isna().mean()).round(1)}).query("missing > 0")

In [ ]:
cols = ["Name", "Age", "SibSp", "Parch", "Pclass", "Ticket", "Fare"]
alone = df[(df["Age"] < 14) & (df["SibSp"] + df["Parch"] == 0)]
print("Minors travelling with SibSp + Parch = 0, and ALL the passengers holding the same ticket:")
display(df[df["Ticket"].isin(alone["Ticket"])].sort_values("Ticket")[cols])
print("Passengers with Fare = 0:")
display(df[df["Fare"] == 0][cols].sort_values("Ticket"))

###  Questions — quality checks
1. Slides 17–18 list six quality dimensions: **accuracy, completeness, consistency, uniqueness, timeliness, semantic validity**. Complete the table, choosing for each check the dimension it verifies:

   | Check | Quality dimension |
   |---|---|
   | duplicated rows / duplicated names | |
   | `Fare = 0` | |
   | `Age < 1` (stored as fractions of a year) | |
   | `Age` of the form xx.5 (estimated ages) | |
   | minors travelling with `SibSp + Parch = 0` | |
   | same ticket, different `Fare` | |
   | missing values | |

2. Three children under 14 travel with `SibSp + Parch = 0`, i.e., apparently without parents. Look at the first table of the cell above (the children and the other passengers holding the **same ticket**). Is this an error in the data? Explain.
3. 15 passengers have `Fare = 0`. Look at their `Ticket` and `Pclass` in the second table. What would you do: (a) replace 0 with another value, (b) keep 0 and add a binary attribute `FreeTicket`, (c) delete these rows? Why?
4. The checks only look at the 891 rows we have. About 1,300 passengers and 900 crew members were on board. Which quality problem can **no check on these rows** reveal?

## 2.2 Profiling missingness: MCAR, MAR or MNAR? (slide 21)

Recall the three mechanisms:
* **MCAR**: the value is missing *by accident*; the probability of being missing is the same for every passenger;
* **MAR**: the probability of being missing depends on **other, observed** attributes (e.g. the class);
* **MNAR**: the probability of being missing depends on **the missing value itself** (e.g. older passengers more often without age).

A first, simple check: if `Age` were MCAR, the share of missing ages would be (about) the same in every group of passengers. The heatmap shows where the missing values are; the table and the chart compare the share of missing values across groups (dashed line = share on all passengers).

In [ ]:
miss_cols = ["Age", "Cabin", "Embarked"]
srt = train_df.sort_values(["Pclass", "Fare"]).reset_index(drop=True)
fig, ax = plt.subplots(figsize=(12, 2.4))
sns.heatmap(srt[miss_cols].isna().T, cbar=False, cmap=["#EEEEEE", "#C00000"], ax=ax)
ax.set_yticklabels(miss_cols, rotation=0)
ax.set_xticks([]); ax.set_xlabel("training passengers sorted by Pclass and Fare →  (red = missing)")
plt.show()

In [ ]:
# Share of passengers with a missing value, in each group (training set)
tr = train_df.assign(Age_missing=train_df["Age"].isna(), Cabin_missing=train_df["Cabin"].isna())
groups = ["Pclass", "Sex", "Embarked", CLASS]
rates = pd.concat({c: tr.groupby(c)[["Age_missing", "Cabin_missing"]].mean() for c in groups}).round(2)
display(rates.style.format("{:.0%}").background_gradient(cmap="Reds", vmin=0, vmax=1))

fig, axes = plt.subplots(1, 4, figsize=(16, 3.2), sharey=True)
for ax, c in zip(axes, groups):
    r = tr.groupby(c)["Age_missing"].mean()
    ax.bar(r.index.astype(str), r.values, color="#C00000")
    ax.axhline(tr["Age_missing"].mean(), ls="--", c="k", lw=1)
    ax.set_title(f"missing Age by {c}"); ax.set_ylim(0, 0.7)
axes[0].set_ylabel("share with missing Age"); axes[0].text(-0.4, tr["Age_missing"].mean() + 0.02, "overall", fontsize=8)
plt.tight_layout(); plt.show()

###  Questions — missingness mechanisms
1. Is `Age` missing completely at random (MCAR)? Answer by comparing the share of missing ages in the groups of `Pclass` and `Embarked` with the overall share (dashed line).
2. Can the data tell us whether `Age` is also MNAR, i.e., whether the probability of being missing depends on the (unknown) age itself? Why?
3. `Cabin` is missing for most 2nd- and 3rd-class passengers, and much more often for those who died (89%) than for survivors (59%). Which mechanism is this? What does it imply for a model that uses "cabin known" as a feature?

## 2.3 Imputation strategies and their side effects (slides 22–24)

We impute `Age` on the training set with several strategies and look at what each strategy does to the **distribution** and to the **relationships** with the other attributes.

In [ ]:
tr = train_df.copy()
tr["Title"] = add_title(tr)
tr["Sex_f"] = (tr["Sex"] == "female").astype(int)
aux = ["Pclass", "Sex_f", "SibSp", "Parch", "Fare"]          # attributes used by the model-based imputers
miss = tr["Age"].isna()

imputed = {
    "Observed only": tr.loc[~miss, "Age"],
    "Mean": tr["Age"].fillna(tr["Age"].mean()),
    "Median": tr["Age"].fillna(tr["Age"].median()),
    "Group median (Title)": tr["Age"].fillna(tr.groupby("Title")["Age"].transform("median")),
    "KNN (k=5)": pd.Series(KNNImputer(n_neighbors=5).fit_transform(tr[["Age"] + aux])[:, 0], index=tr.index),
    "Iterative (Random Forest)": pd.Series(IterativeImputer(estimator=RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE),
                                                            max_iter=10, random_state=RANDOM_STATE).fit_transform(tr[["Age"] + aux])[:, 0], index=tr.index),
}
summary = pd.DataFrame({k: {"mean": v.mean(), "std": v.std(), "corr(Age, Pclass)": v.corr(tr.loc[v.index, "Pclass"]),
                            "corr(Age, SibSp)": v.corr(tr.loc[v.index, "SibSp"])} for k, v in imputed.items()}).T.round(3)
summary

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 6.5), sharex=True)
bins = np.arange(0, 82, 2)
for ax, (name, v) in zip(axes.ravel(), imputed.items()):
    if name == "Observed only":
        ax.hist(v, bins=bins, color="#7F7F7F")
    else:
        ax.hist([v[~miss], v[miss]], bins=bins, stacked=True, color=["#7F7F7F", "#C00000"], label=["observed", "imputed"])
        ax.legend(fontsize=9)
    ax.set_title(f"{name}  (std = {v.std():.1f})")
plt.suptitle("Age in the training set: observed values (grey) and imputed values (red)", y=1.01)
plt.tight_layout(); plt.show()

###  Questions — side effects of imputation
1. Which strategies reduce the standard deviation of `Age`? Explain the spike in the mean/median histograms (slide 24).
2. How does mean imputation change the correlation between `Age` and `Pclass`? Why?
3. Group-median imputation by `Title` is a simple form of *regression imputation*: which information does `Title` carry about age?
4. Every imputation injects an assumption (slide 23). Write the assumption made by KNN imputation and by group-median imputation.
5. The model-based imputers use `Fare`, `SibSp`, `Parch`, `Pclass` and `Sex`. Would it be correct to add `Survived` to the attributes used for imputation?

## 2.4 Does the imputation strategy change the model? (slides 25–26)

We now impute `Age` **inside a pipeline** with four imputers (mean, median, KNN, random forest) and train two classifiers (Logistic Regression and Random Forest).

To compare them in a simple way we **count passengers**. We use `cross_val_predict` with 5 folds: every training passenger is predicted exactly once, by a model that was trained (imputer included) on the other 4 folds. For each combination the table reports:
* the accuracy and the **number of passengers classified correctly** (out of 712);
* the number of correct predictions among the **137 passengers whose age is missing** (the only ones whose input changes with the imputer);
* how many predictions are **different from those obtained with the mean imputer**.

In [ ]:
NUM = ["Age", "Fare", "SibSp", "Parch", "Pclass"]

def make_prep(age_imputer="median"):
    """Preparation: impute + scale the numerical attributes, impute + one-hot the categorical ones."""
    imp = {"mean": SimpleImputer(strategy="mean"),
           "median": SimpleImputer(strategy="median"),
           "knn": KNNImputer(n_neighbors=5),
           "random forest": IterativeImputer(estimator=RandomForestRegressor(n_estimators=50, random_state=RANDOM_STATE),
                                             max_iter=5, random_state=RANDOM_STATE)}[age_imputer]
    return ColumnTransformer([
        ("num", make_pipeline(imp, StandardScaler()), NUM),
        ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), ["Sex", "Embarked"])])

IMPUTERS = ["mean", "median", "knn", "random forest"]
CLASSIFIERS = {"Logistic Regression": LogisticRegression(max_iter=1000),
               "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)}
y_tr = train_df[CLASS].values
age_missing = train_df["Age"].isna().values
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rows, preds = [], {}
for m_name, model in CLASSIFIERS.items():
    for imp in IMPUTERS:
        # every training passenger is predicted exactly once, by a model trained on the other 4 folds
        p = cross_val_predict(Pipeline([("prep", make_prep(imp)), ("model", model)]), train_df, y_tr, cv=cv5)
        preds[(m_name, imp)] = p
        rows.append({"Classifier": m_name, "Age imputer": imp,
                     "accuracy": (p == y_tr).mean(),
                     f"correct (of {len(y_tr)})": int((p == y_tr).sum()),
                     f"correct among the {age_missing.sum()} with missing Age": int((p == y_tr)[age_missing].sum()),
                     "predictions different from 'mean'": int((p != preds[(m_name, "mean")]).sum())})
imp_res = pd.DataFrame(rows).set_index(["Classifier", "Age imputer"])
display(imp_res.style.format({"accuracy": "{:.3f}"}).background_gradient(cmap="Greens", subset=["accuracy"]))

fig, ax = plt.subplots(figsize=(9, 3.5))
imp_res[f"correct (of {len(y_tr)})"].unstack(0).reindex(IMPUTERS).plot(kind="bar", ax=ax, rot=0, color=["#4472C4", "#2E9E8F"])
for cont in ax.containers: ax.bar_label(cont, fontsize=9)
ax.set_ylim(500, 615); ax.legend(ncol=2, loc="upper center"); ax.set_ylabel("passengers classified correctly"); ax.set_title("Cross-validated predictions on the 712 training passengers")
plt.show()

###  Questions — imputation and model performance
1. For each classifier, how many passengers separate the best and the worst imputer? Is this a large difference, out of 712 passengers?
2. How many predictions change when we replace the mean imputer with the others? Given the classifier, why does the imputer have so little effect on the results? (Think of how many passengers are affected and of how much `Age` matters once `Sex` and `Pclass` are known.)
3. The Random Forest classifier is (slightly) better than Logistic Regression with every imputer. Why?
4. An alternative to imputation is to drop the 137 training passengers with a missing age. Why is it a dangerous choice for this dataset? (Recall 2.2.)

## 2.5 A controlled experiment: MCAR, MAR and MNAR (as in the Ames Housing example, slides 27–35)

With real missing values we never know the true value, so we cannot say how good an imputation is. Here we **create** the missing values ourselves:
1. we take the training passengers whose age **is known** (575);
2. we delete a share `RATE` (30%) of their ages with each of the three mechanisms of slide 21:
   * **MCAR**: every passenger has the same probability of losing the age;
   * **MAR**: 3rd-class passengers lose the age more often (the probability depends on `Pclass`, an observed attribute);
   * **MNAR**: older passengers lose the age more often (the probability depends on the age itself);
3. we impute the deleted ages with four imputers and **compare the imputed values with the true ones**.

The cell prints three tables.

**(1) Are the remaining ages representative of the deleted ones?** For each mechanism:
* *true mean of the deleted ages*: the average real age of the passengers whose age we deleted. In real life we never know it; here we do;
* *mean of the remaining ages*: the average of the ages still visible. This is exactly the value that mean imputation writes in every hole.

If the two numbers are close, the visible data are representative of the missing ones; if they differ (e.g. 27.1 vs 31.0), the passengers who lost their age are different from the others, and any imputer that ignores *why* they are missing will be wrong on average.

**(2) RMSE** (root mean squared error), computed on the deleted passengers only:
$\text{RMSE} = \sqrt{\text{average of } (\text{imputed} - \text{true})^2}$. It is the **typical size of the error, in years**, whatever its sign: RMSE = 13 means that an imputed age is typically about 13 years away from the truth.

**(3) Bias**: $\text{bias} = \text{average of } (\text{imputed} - \text{true})$. It tells whether the imputer makes a **systematic** error: bias > 0 → it overestimates the ages on average; bias < 0 → it underestimates them; bias ≈ 0 → the errors cancel out on average.

A small example shows the difference: if the true ages are 20 and 40 and both are imputed as 30, the bias is 0 (the average is right) but the RMSE is 10 (each single value is 10 years off). A good imputer has both a small bias and a small RMSE.

In [ ]:
RATE = 0.30          #  share of the known ages that we delete

known = train_df.dropna(subset=["Age"]).reset_index(drop=True)
known["Title"] = add_title(known); known["Sex_f"] = (known["Sex"] == "female").astype(int)
true_age = known["Age"].values
rng = np.random.default_rng(RANDOM_STATE)

def mask(mechanism):
    n = len(known)
    if mechanism == "MCAR":                       # completely at random
        p = np.full(n, RATE)
    elif mechanism == "MAR":                      # depends on an observed attribute: 3rd class lost more often
        w = known["Pclass"].map({1: 0.3, 2: 0.6, 3: 1.6}).values; p = RATE * w / w.mean()
    else:                                         # MNAR: depends on the age itself, older passengers lost more often
        w = np.exp((true_age - true_age.mean()) / 12); p = RATE * w / w.mean()
    return rng.random(n) < np.clip(p, 0, 1)

def impute(d, method):
    if method == "Mean": return d["Age"].fillna(d["Age"].mean())
    if method == "Group median (Title)": return d["Age"].fillna(d.groupby("Title")["Age"].transform("median"))
    cols = ["Age", "Pclass", "Sex_f", "SibSp", "Parch", "Fare"]
    imp = KNNImputer(n_neighbors=5) if method == "KNN" else IterativeImputer(
        estimator=RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE), max_iter=10, random_state=RANDOM_STATE)
    return pd.Series(imp.fit_transform(d[cols])[:, 0])

METHODS = ["Mean", "Group median (Title)", "KNN", "Iterative (RF)"]
means, rmse, bias = {}, {}, {}
for mech in ["MCAR", "MAR", "MNAR"]:
    m = mask(mech)
    d = known.copy(); d.loc[m, "Age"] = np.nan
    means[mech] = {"deleted ages": int(m.sum()), "true mean of the deleted ages": true_age[m].mean(),
                   "mean of the remaining ages (= value used by mean imputation)": d["Age"].mean()}
    rmse[mech], bias[mech] = {}, {}
    for method in METHODS:
        err = impute(d, method).values[m] - true_age[m]          # imputed − true, only on the deleted passengers
        rmse[mech][method] = np.sqrt(np.mean(err ** 2))
        bias[mech][method] = err.mean()

print("(1) Are the remaining ages representative of the deleted ones?")
display(pd.DataFrame(means).round(1))
print("(2) RMSE in years: typical size of the error (lower = better)")
display(pd.DataFrame(rmse).round(1).style.format("{:.1f}").background_gradient(cmap="Reds", axis=None))
print("(3) Bias in years: average of (imputed − true); > 0 overestimates, < 0 underestimates, 0 = no systematic error")
display(pd.DataFrame(bias).round(1).style.format("{:+.1f}").background_gradient(cmap="RdBu_r", vmin=-20, vmax=20, axis=None))

###  Questions — controlled experiment
1. Look at table (1). Under MCAR, is the mean of the remaining ages a good estimate of the true mean of the deleted ones? And under MAR and MNAR? Explain why.
2. Which imputer has the lowest RMSE under each mechanism? Why does using other attributes reduce the bias under MAR?
3. Under MNAR all imputers have a large negative bias. What does a negative bias mean here, and why can no imputer based on the observed data remove it (slide 21)?
4.  Change `RATE` to 0.10 and then to 0.50 and re-run the cell. How do RMSE and bias change? Does deleting fewer values make MNAR less harmful?
5. In the real Titanic data (2.2), which of the three scenarios do you think is closest to reality for `Age`?

---
# Part 3 — Scaling (slides 36–47)

## 3.1 Applying the three scalers

The scalers learn their parameters **on the training set**. We look at `Age` and `Fare` (which has very large fares, up to 512 £).

In [ ]:
num2 = ["Age", "Fare"]
Xs_tr = train_df[num2].fillna(train_df[num2].median())
scalers = {"MinMax": MinMaxScaler(), "Robust": RobustScaler(), "Z-score": StandardScaler()}
params = {
    "MinMax": lambda s: {f"{c}: x_min, x_max": (round(s.data_min_[i], 2), round(s.data_max_[i], 2)) for i, c in enumerate(num2)},
    "Robust": lambda s: {f"{c}: Q2, Q3-Q1": (round(s.center_[i], 2), round(s.scale_[i], 2)) for i, c in enumerate(num2)},
    "Z-score": lambda s: {f"{c}: μ, δ": (round(s.mean_[i], 2), round(s.scale_[i], 2)) for i, c in enumerate(num2)},
}
scaled = {"Raw": Xs_tr}
for name, sc in scalers.items():
    sc.fit(Xs_tr)
    print(f"{name:8s}", params[name](sc))
    scaled[name] = pd.DataFrame(sc.transform(Xs_tr), columns=num2)
pd.concat({k: v.describe().loc[["min", "25%", "50%", "75%", "max"]] for k, v in scaled.items()}, axis=1).round(2)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.6))
for ax, (name, d) in zip(axes, scaled.items()):
    sns.boxplot(data=d, orient="h", ax=ax, palette=["#7F7F7F", "#4472C4"])
    ax.set_title(name)
plt.suptitle("Age and Fare before and after scaling (training set)", y=1.03); plt.tight_layout(); plt.show()

## 3.2 Effect on distances (slides 36, 41)

In [ ]:
# three passengers of the training set: same Fare and different Age vs same Age and different Fare
p = Xs_tr.iloc[[0]].copy()
cands = Xs_tr.assign(dAge=(Xs_tr["Age"] - p["Age"].iloc[0]).abs(), dFare=(Xs_tr["Fare"] - p["Fare"].iloc[0]).abs())
p2 = cands[(cands.dFare < 1) & (cands.dAge > 15)].iloc[[0]][num2]            # similar Fare, very different Age
p3 = cands[(cands.dAge < 1) & (cands.dFare > 15)].iloc[[0]][num2]            # similar Age, very different Fare
trio = pd.concat([p, p2, p3]); trio.index = ["p1", "p2", "p3"]
display(trio)
rows = {}
for name, sc in [("Raw", None)] + list(scalers.items()):
    t = trio.values if sc is None else sc.transform(trio)
    rows[name] = {"d(p1,p2)": np.linalg.norm(t[0] - t[1]), "d(p1,p3)": np.linalg.norm(t[0] - t[2])}
pd.DataFrame(rows).T.round(3)

## 3.3 Effect on models: KNN vs Decision Tree (slide 37)

The indicator reported is the **accuracy** (share of passengers classified correctly), averaged over a 5-fold cross-validation repeated 3 times on the training set.

In [ ]:
K_KNN = 7            #  number of neighbours of the KNN classifier

def scaling_prep(scaler):
    num = ["Age", "Fare", "SibSp", "Parch", "Pclass"]
    steps = [SimpleImputer(strategy="median")] + ([scaler] if scaler is not None else [])
    return ColumnTransformer([("num", make_pipeline(*steps), num),
                              ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), ["Sex", "Embarked"])])

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE)
rows = []
for s_name, sc in {"No scaling": None, "MinMax": MinMaxScaler(), "Robust": RobustScaler(), "Z-score": StandardScaler()}.items():
    for m_name, m in {f"KNN (k={K_KNN})": KNeighborsClassifier(n_neighbors=K_KNN), "Decision Tree (depth 5)": DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE)}.items():
        s = cross_val_score(Pipeline([("prep", scaling_prep(sc)), ("model", m)]), train_df, train_df[CLASS], cv=cv, scoring="accuracy")
        rows.append({"Scaler": s_name, "Model": m_name, "accuracy": round(s.mean(), 3)})
print("Indicator: ACCURACY = share of passengers classified correctly")
print("(mean over 5-fold cross-validation repeated 3 times on the training set)")
pd.DataFrame(rows).pivot(index="Scaler", columns="Model", values="accuracy")

## 3.4 The impact of an outlier (slides 39–41)

We add one extreme fare (a data-entry error: 5,000 instead of 50) and look at what each scaler does to the **other** passengers. The table reports the interquartile range (IQR) of the scaled fares, i.e., how spread out the bulk of the passengers remains after scaling.

In [ ]:
# add one extreme fare (a data-entry error: 5,000 instead of 50) and look at what each scaler does to the other passengers
X_out = pd.concat([Xs_tr, pd.DataFrame({"Age": [30], "Fare": [5000]})], ignore_index=True)
rows = {}
for name, sc in scalers.items():
    f_before = sc.fit(Xs_tr).transform(Xs_tr)[:, 1]
    f_after = sc.fit(X_out).transform(X_out)[:-1, 1]
    rows[name] = {"Fare' IQR without outlier": np.subtract(*np.percentile(f_before, [75, 25])),
                  "Fare' IQR with outlier": np.subtract(*np.percentile(f_after, [75, 25])),
                  "Fare' of the outlier": sc.transform(X_out)[-1, 1]}
pd.DataFrame(rows).T.round(3)

###  Questions — scaling
1. For each scaler, write how the passenger with `Fare` = 512.33 is transformed, using the parameters printed in 3.1. Which scaler keeps the bulk of the fares spread out?
2. In 3.2, compare d(p1,p2) and d(p1,p3) before and after scaling. Which feature dominates the raw distance and why (slide 36)?
3. In 3.3, KNN improves with scaling while the Decision Tree does not change. Explain why using the sensitivity table of slide 37.
4.  In 3.3 set `K_KNN = 1` and then `K_KNN = 25` and re-run the cell. Does scaling still help KNN? Does the best scaler change?
5. In 3.4, after adding the outlier, how much does the interquartile range of the scaled fares shrink with MinMax and Z-score? Why is the Robust scaler unaffected? What is its drawback?
6. `Fare` is strongly skewed. Would you apply another transformation before scaling? Which one, and why?

---
# Part 4 — Sampling and selection bias (slides 48–62)

## 4.1 Random vs stratified sampling (slides 60–62)

We treat the 889 passengers with a known port as the **population** and draw many small samples (`FRAC` = 5%, i.e., 44 passengers). How well does each sample represent the population? We look at the share of passengers embarked in Queenstown (Q), a rare category (8.7%).

In [ ]:
FRAC = 0.05          #  sample size, as a fraction of the 889 passengers
RUNS = 300

pop = df.dropna(subset=["Embarked"]).reset_index(drop=True)
share_rand, share_strat = [], []
for r in range(RUNS):
    s_r = pop.sample(frac=FRAC, random_state=r)
    s_s, _ = train_test_split(pop, train_size=FRAC, stratify=pop["Embarked"], random_state=r)
    share_rand.append((s_r["Embarked"] == "Q").mean()); share_strat.append((s_s["Embarked"] == "Q").mean())
share_rand, share_strat = np.array(share_rand), np.array(share_strat)
true_q = (pop["Embarked"] == "Q").mean()

fig, ax = plt.subplots(figsize=(10, 3.5))
bins = np.linspace(0, max(0.26, share_rand.max() + 0.02), 25)
ax.hist([share_rand, share_strat], bins=bins, color=["#7F7F7F", "#ED7D31"], label=["random sampling", "stratified sampling (by Embarked)"])
ax.axvline(true_q, color="k", ls="--", label=f"population share = {true_q:.3f}")
ax.set_xlabel("share of Queenstown passengers in the sample"); ax.set_ylabel("number of samples"); ax.legend()
ax.set_title(f"{RUNS} samples of {round(FRAC * len(pop))} passengers each")
plt.show()
print(f"std of the share — random: {share_rand.std():.3f}   stratified: {share_strat.std():.3f}")
print(f"samples with NO Queenstown passenger — random: {(share_rand == 0).sum()} of {RUNS}   stratified: {(share_strat == 0).sum()} of {RUNS}")

## 4.2 Simulating a coverage bias (slides 50–52)

The **target population** is all passengers. The **sampling frame**, however, contains only 2nd- and 3rd-class passengers (e.g., the passenger lists of 1st class were not accessible). We train a model on the frame and evaluate it on the test set (which represents the target population), separately for each class.

In [ ]:
frame = train_df[train_df["Pclass"] != 1]
repr_sample = train_df.sample(n=len(frame), random_state=RANDOM_STATE)        # a representative sample of the same size

def fit_eval(train):
    pipe = Pipeline([("prep", make_prep("median")), ("model", RandomForestClassifier(n_estimators=300, max_depth=6, random_state=RANDOM_STATE))])
    pipe.fit(train, train[CLASS]); pred = pipe.predict(test_df); proba = pipe.predict_proba(test_df)[:, 1]
    res = {"share of 1st class in training": (train["Pclass"] == 1).mean(), "accuracy: overall": accuracy_score(test_df[CLASS], pred)}
    for c in (1, 2, 3):
        m = test_df["Pclass"] == c; res[f"accuracy: Pclass {c}"] = accuracy_score(test_df.loc[m, CLASS], pred[m])
    m = test_df["Pclass"] == 1
    res["1st class: mean predicted P(survive)"] = proba[m].mean()
    res["1st class: true survival rate"] = test_df.loc[m, CLASS].mean()
    m = (test_df["Pclass"] == 1) & (test_df["Sex"] == "male")
    res["1st-class men: predicted survival rate"] = pred[m].mean()
    res["1st-class men: true survival rate"] = test_df.loc[m, CLASS].mean()
    return pd.Series(res)

pd.DataFrame({"trained on the biased frame": fit_eval(frame), "trained on a representative sample": fit_eval(repr_sample)}).round(3)

## 4.3 Detecting the coverage bias (slide 58)

Coverage bias is detected by comparing the distributions of the dataset with those of the target population along the key dimensions (ratio 1 = same share; red = under-represented, blue = over-represented).

In [ ]:
dims = {"Pclass": df["Pclass"], "Sex": df["Sex"], "Embarked": df["Embarked"], "Age band": pd.cut(df["Age"], [0, 12, 18, 40, 60, 80])}
rows = []
for name, col in dims.items():
    tgt = col.value_counts(normalize=True)
    smp = col.loc[df["PassengerId"].isin(frame["PassengerId"])].value_counts(normalize=True)
    for v in tgt.index:
        rows.append({"dimension": name, "value": str(v), "target population": tgt[v], "frame": smp.get(v, 0.0)})
cmp_ = pd.DataFrame(rows); cmp_["ratio frame/target"] = cmp_["frame"] / cmp_["target population"]
cmp_.round(3).style.format(precision=3).background_gradient(cmap="RdBu", subset=["ratio frame/target"], vmin=0, vmax=2)

###  Questions — sampling and bias
1. In 4.1, how much does the share of Queenstown passengers vary across random samples? How many random samples miss this category completely? Why does stratified sampling never miss it (slide 62)?
2.  Set `FRAC = 0.02` (18 passengers) and then `FRAC = 0.20` (178 passengers) and re-run 4.1. How do the variability and the number of samples without Queenstown passengers change? When is stratification most useful?
3. Stratified sampling here uses `Embarked`. Would it also guarantee the right share of survivors? What would you stratify on to build a train/test split for this task?
4. In 4.2, the model trained on the biased frame has a similar (here even slightly higher) overall accuracy. Why is the overall accuracy misleading here? Compare the mean predicted survival probability of 1st-class passengers with their true survival rate. Look at 1st-class men: what does the model predict for them, and why?
5. In 4.3, which dimensions reveal the bias? Which bias would remain *invisible* with this check?
6. Map this simulation onto one of the bias examples of slides 51–57. How would you mitigate the bias (slide 59)?

---
# Part 5 — Mitigating the coverage bias: collecting data vs oversampling (slides 59, 65)

In 4.2 the training frame contained **no** 1st-class passenger. Slide 59 says that coverage bias is addressed during data collection: suppose that we manage to recover the records of **half** of the 1st-class passengers (`RESTORE = 0.5`). We then have two options:
* use the recovered passengers as they are: 1st class is present, but it is only 13.7% of the training set instead of 24%;
* restore the original share with **random oversampling** (slide 65): duplicate recovered 1st-class passengers, chosen at random, until 1st class has its original size.

We compare four training sets, using the same model as in 4.2 and the same test set:

| Training set | 1st-class passengers |
|---|---|
| 0% (biased frame) | none |
| 50% recovered | half of them, real |
| 50% recovered + oversampled to 100% | half of them, real, plus duplicates up to the original number |
| original training set | all of them, real |

Which 1st-class passengers are recovered is a matter of chance, so the two "recovered" scenarios are repeated `REPEATS` = 10 times with different random choices, and the table reports the average.

In [ ]:
RESTORE = 0.5        #  fraction of the 1st-class training passengers that we manage to recover
REPEATS = 10         # the recovered passengers are chosen at random: we repeat and average

first = train_df[train_df["Pclass"] == 1]
print(f"1st-class passengers in the training set: {len(first)} of {len(train_df)} ({len(first) / len(train_df):.1%})")

scenarios = {"0% (biased frame)": [fit_eval(frame)]}
restored, oversampled = [], []
for r in range(REPEATS):
    recovered = first.sample(frac=RESTORE, random_state=r)                       # the 1st-class passengers we recover
    part = pd.concat([frame, recovered])
    restored.append(fit_eval(part))
    # random oversampling: duplicate recovered passengers until 1st class has its original size (and share)
    copies = recovered.sample(n=len(first) - len(recovered), replace=True, random_state=r)
    oversampled.append(fit_eval(pd.concat([part, copies])))
scenarios[f"{RESTORE:.0%} recovered"] = restored
scenarios[f"{RESTORE:.0%} recovered + oversampled to 100%"] = oversampled
scenarios["original training set (100% real)"] = [fit_eval(train_df)]
mitig = pd.DataFrame({k: pd.concat(v, axis=1).mean(axis=1) for k, v in scenarios.items()})
display(mitig.round(3))

fig, axes = plt.subplots(1, 2, figsize=(14, 3.8))
labels = [k.replace(" (", "\n(").replace(" + ", "\n+ ") for k in mitig.columns]
cols = ["#7F7F7F", "#4472C4", "#ED7D31", "#2E9E8F"]
axes[0].bar(labels, mitig.loc["1st class: mean predicted P(survive)"], color=cols)
axes[0].axhline(mitig.loc["1st class: true survival rate"].iloc[0], ls="--", c="k", label="true survival rate")
axes[0].set_title("1st class: mean predicted P(survive)"); axes[0].set_ylim(0, 0.75); axes[0].legend(loc="upper left")
axes[1].bar(labels, mitig.loc["accuracy: Pclass 1"], color=cols); axes[1].set_ylim(0.5, 0.85)
axes[1].set_title("accuracy on the 45 1st-class test passengers")
for ax in axes:
    for cont in ax.containers: ax.bar_label(cont, fmt="%.2f", label_type="center", color="white", fontweight="bold")
    ax.tick_params(axis="x", labelsize=9)
plt.tight_layout(); plt.show()

###  Questions — mitigating the coverage bias
1. Compare the overall accuracy of the four training sets. Does it reveal the differences between them? Why not?
2. Compare the mean predicted survival probability of 1st-class passengers with their true survival rate (0.56). How much of the gap of the biased frame is closed by recovering half of the 1st-class passengers?
3. Oversampling brings the share of 1st class back to 24%. Does it improve the predictions for 1st-class passengers with respect to "50% recovered"? Look at the accuracy on 1st class and at the predicted survival rate of 1st-class men (true rate 0.37). Why can duplicated passengers not replace the missing real ones (slide 65)?
4.  Set `RESTORE = 0.25` and then `RESTORE = 0.75` and re-run the cell. How quickly does the model approach the one trained on the original training set? What does this suggest about how much data must be collected to mitigate a coverage bias?

---
# Part 6 — Wrap-up: an end-to-end preparation pipeline

We combine the choices of the previous parts into a single pipeline, select the model with cross-validation on the training set, compare it with a **baseline** (the minimal preparation used in 2.4: median imputation, scaling, one-hot encoding of `Sex` and `Embarked`, with Logistic Regression), and finally evaluate it **once** on the test set put aside in the Setup.

In [ ]:
def add_features(X):
    X = X.copy()
    X["Title"] = add_title(X)
    X["CabinKnown"] = X["Cabin"].notna().astype(int)
    X["FamilySize"] = X["SibSp"] + X["Parch"] + 1
    X["LogFare"] = np.log1p(X["Fare"])
    return X

num_f = ["Age", "LogFare", "FamilySize", "Pclass", "CabinKnown"]
cat_f = ["Sex", "Embarked", "Title"]
final_prep = Pipeline([
    ("features", FunctionTransformer(add_features)),
    ("columns", ColumnTransformer([
        ("num", make_pipeline(IterativeImputer(estimator=RandomForestRegressor(n_estimators=50, random_state=RANDOM_STATE), max_iter=5,
                                               random_state=RANDOM_STATE, add_indicator=True), RobustScaler()), num_f),
        ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), cat_f)]))])

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE)
# baseline: the minimal preparation (median imputation + scaling + one-hot of Sex and Embarked) with logistic regression
baseline = cross_val_score(Pipeline([("prep", make_prep("median")), ("model", LogisticRegression(max_iter=1000))]),
                           train_df, train_df[CLASS], cv=cv, scoring="accuracy")

candidates = {"Logistic Regression": LogisticRegression(max_iter=1000), "KNN (k=9)": KNeighborsClassifier(n_neighbors=9),
              "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=6, random_state=RANDOM_STATE)}
rows = []
for name, m in candidates.items():
    s = cross_val_score(Pipeline([("prep", final_prep), ("model", m)]), train_df, train_df[CLASS], cv=cv, scoring="accuracy")
    rows.append({"model": name, "CV accuracy (mean)": s.mean(), "CV accuracy (std)": s.std()})
cv_res = pd.DataFrame(rows).set_index("model").round(3)
display(cv_res)
print(f"Baseline (minimal preparation + Logistic Regression): {baseline.mean():.3f}")
print(f"Majority-class accuracy (always 'Died'): {1 - train_df[CLASS].mean():.3f}")

In [ ]:
best = cv_res["CV accuracy (mean)"].idxmax()
final = Pipeline([("prep", final_prep), ("model", candidates[best])]).fit(train_df, train_df[CLASS])
pred = final.predict(test_df)
print(f"Selected model: {best}")
print(f"Test accuracy (computed once!): {accuracy_score(test_df[CLASS], pred):.3f}")
print(classification_report(test_df[CLASS], pred, target_names=["Died", "Survived"]))

###  Questions — wrap-up
1. List the preparation steps of the final pipeline and, for each one, the part of the lab (and the slide) that motivated it.
2. How much does the final pipeline improve over the baseline? Which step do you think contributes most? Propose an experiment to verify it (an *ablation*: remove one step at a time).
3. Is the test accuracy close to the cross-validation estimate? Why must the test set be used only once?
4. Which preparation choices of this lab would you revisit if the model had to be deployed on passengers of a *different* ship? Relate your answer to the sampling and bias part.
5. Summing up: write three "rules of thumb" on data preparation that you learnt in this lab.